# CITD Image Processing — OCR Recognition Fine-Tuning (Colab)

Fine-tunes PaddleOCR's recognition model on Vietnamese plate crops using PaddleX.
This notebook does **not** fetch the dataset itself — upload the PaddleX dataset
produced locally by `scripts/prepare-ocr-dataset.py` (an `images/` folder plus
`train.txt`/`val.txt`, zipped) either via Google Drive or the upload widget below.

Dataset source, license caveat (no LICENSE file on the upstream GitHub repo —
academic/non-commercial use only), and the pinned commit are documented in
`docs/ocr-dataset-selection.md`. Do not commit the uploaded dataset or the
trained model weights back into the repository.

Runtime → Change runtime type → select a GPU before running this notebook.

In [1]:
# Fail fast with a clear message instead of training on CPU by accident.
import subprocess

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(
        "No GPU detected. Use Runtime \u2192 Change runtime type \u2192 GPU, then rerun."
    )
print(result.stdout)

Wed Sep 16 16:31:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# paddlepaddle-gpu is not on PyPI and its wheel bundles CUDA/cuDNN (~3GB), so
# downloading it fresh every Colab session is slow. Cache the .whl on Drive once
# and reuse it on later runs instead of re-downloading from PaddlePaddle's CDN.
import re
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
CACHE_DIR = Path("/content/drive/MyDrive/CITD/XuLyAnh/Doan/pip-cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

cached_wheels = list(CACHE_DIR.glob("paddlepaddle_gpu*.whl"))
if cached_wheels:
    wheel_path = cached_wheels[0]
    print(f"Installing paddlepaddle-gpu from Drive cache: {wheel_path.name}")
else:
    import subprocess

    smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
    match = re.search(r"CUDA Version:\s*([\d.]+)", smi)
    if not match:
        raise RuntimeError("Could not detect CUDA version from nvidia-smi output")
    cuda_version = match.group(1)
    major, minor = (int(part) for part in cuda_version.split(".")[:2])
    channels = [129, 126, 123, 120, 118]
    detected = major * 10 + minor
    channel = next((c for c in channels if c <= detected), channels[-1])
    index_url = f"https://www.paddlepaddle.org.cn/packages/stable/cu{channel}/"
    print(f"No cached wheel found; downloading from {index_url} (first time only, ~3GB)")
    # %pip (unlike subprocess.run) renders pip's progress bar live in the cell output.
    %pip download --timeout 1000 --retries 10 -d {CACHE_DIR} paddlepaddle-gpu -i {index_url}
    cached_wheels = list(CACHE_DIR.glob("paddlepaddle_gpu*.whl"))
    assert cached_wheels, "pip download did not produce a paddlepaddle-gpu wheel"
    wheel_path = cached_wheels[0]

%pip install -q {wheel_path}
%pip install -q --timeout 1000 --retries 10 "paddlex[ocr]"

Mounted at /content/drive
Installing paddlepaddle-gpu from Drive cache: paddlepaddle_gpu-3.3.1-cp313-cp313-linux_x86_64.whl
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 580.7/580.7 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 781.4/781.4 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.8/331.8 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.4/366.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 MB 4.7 MB/s eta 0:00:00
   

In [3]:
# paddlepaddle-gpu and Colab's preinstalled torch hard-pin mutually incompatible
# nvidia-nccl-cu12/nvidia-cudnn-cu12 versions; whichever is installed last breaks the
# other. paddlex only needs modelscope as an optional model-download source, which
# unconditionally imports torch. Stub modelscope out (empty module) so paddlex's
# `import modelscope` succeeds without ever pulling torch in.
import site
import subprocess
import sys
from pathlib import Path

subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "modelscope"], check=False
)
site_packages = Path(site.getsitepackages()[0])
stub_dir = site_packages / "modelscope"
stub_dir.mkdir(parents=True, exist_ok=True)
(stub_dir / "__init__.py").write_text("")
print(f"Stubbed modelscope at: {stub_dir}")

Stubbed modelscope at: /usr/local/lib/python3.13/dist-packages/modelscope


In [4]:
# Now that modelscope (and therefore torch) is never imported, restore the exact
# nvidia-nccl-cu12/nvidia-cudnn-cu12 versions paddlepaddle-gpu itself requires,
# since paddle actively uses cudnn for its own GPU operations.
%pip install -q --timeout 1000 --retries 10 \
    "nvidia-nccl-cu12==2.27.3" "nvidia-cudnn-cu12==9.9.0.52"

## Install the PaddleOCR training plugin

`paddlex`'s pip package is inference/serving-only; the real training code
(`tools/train.py`) lives in a separately-installed "plugin" repo. This mirrors
what `scripts/train-ocr.py` does locally, debugged and confirmed working.

In [7]:
import subprocess
import sys
from pathlib import Path

import paddlex

REPO_DIR = Path(paddlex.__file__).parent / "repo_manager" / "repos" / "PaddleOCR"

if not (REPO_DIR / "tools" / "train.py").is_file():
    # paddlex.repo_manager assumes its "repos" parent directory already exists when
    # it tries to clean up a previous (possibly absent) install before cloning.
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(
        [sys.executable, "-m", "paddlex", "--install", "PaddleOCR", "-y"],
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        # This installer shells out to the *system* python (/usr/bin/python3, a
        # separate interpreter/pip from this notebook's kernel) to bulk-install
        # PaddleOCR's *and* PaddleNLP's combined requirements. PaddleNLP pulls in
        # seqeval (NER metrics, unrelated to PaddleOCR's text-recognition
        # training), whose old sdist commonly fails to build under Python 3.13
        # (no stdlib distutils). The repo clone/extract step above already
        # succeeded regardless, so treat this as fatal only if tools/train.py
        # (what we actually need) is still missing afterwards.
        print(
            "\nWARNING: paddlex --install PaddleOCR exited non-zero (see output above). "
            "Continuing since the PaddleOCR repo itself was already cloned -- this is "
            "commonly just seqeval (a PaddleNLP-only dependency) failing to build."
        )

assert (REPO_DIR / "tools" / "train.py").is_file(), f"Missing tools/train.py under {REPO_DIR}"
print(f"PaddleOCR training plugin ready at: {REPO_DIR}")

# Safety net: install the plugin's own requirements directly in case paddlex's
# internal auto-install skipped anything (observed locally for scikit-image).
%pip install -q -r {REPO_DIR}/requirements.txt

PaddleOCR training plugin ready at: /usr/local/lib/python3.13/dist-packages/paddlex/repo_manager/repos/PaddleOCR
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.8/346.8 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 125.1 MB/s eta 0:00:00


## Load the dataset

Pick **one** of the next two cells depending on where you uploaded the dataset
zip (an `images/` folder plus `train.txt`/`val.txt`, matching the output of
`scripts/prepare-ocr-dataset.py --out data/processed/ocr-rec-dataset-merged`
locally -- this merges the original GitHub source with the LPRNet `valid/`
extra source by default; see `docs/ocr-dataset-selection.md`).

In [8]:
# Option A: the zip is already in Google Drive. Set DATASET_ZIP to its path.
from google.colab import drive

drive.mount("/content/drive")
# Merged dataset (original GitHub source + LPRNet valid/ extra source, 557 unique
# plate texts vs. 421 before -- see docs/ocr-dataset-selection.md). Upload
# data/processed/ocr-rec-dataset-merged.zip here before running this cell.
DATASET_ZIP = "/content/drive/MyDrive/CITD/XuLyAnh/Doan/ocr-rec-dataset-merged.zip"  # edit this path

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Option B: upload the zip directly through the browser instead of Drive.
from google.colab import files

uploaded = files.upload()
DATASET_ZIP = next(iter(uploaded))

StopIteration: 

In [9]:
import shutil
import zipfile
from pathlib import Path

DATASET_DIR = Path("/content/ocr-rec-dataset")
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
DATASET_DIR.mkdir(parents=True)

with zipfile.ZipFile(DATASET_ZIP) as archive:
    archive.extractall(DATASET_DIR)

# The zip may contain the dataset directly or wrap it in one extra folder.
if not (DATASET_DIR / "train.txt").is_file():
    nested = [child for child in DATASET_DIR.iterdir() if child.is_dir()]
    if len(nested) == 1 and (nested[0] / "train.txt").is_file():
        DATASET_DIR = nested[0]

train_count = sum(1 for _ in (DATASET_DIR / "train.txt").open(encoding="utf-8"))
val_count = sum(1 for _ in (DATASET_DIR / "val.txt").open(encoding="utf-8"))
print(f"Dataset dir: {DATASET_DIR}")
print(f"train.txt lines: {train_count}")
print(f"val.txt lines: {val_count}")

Dataset dir: /content/ocr-rec-dataset
train.txt lines: 12110
val.txt lines: 1210


In [10]:
# Sanity-check one sample before spending GPU time on a malformed dataset.
from IPython.display import Image, display

with (DATASET_DIR / "train.txt").open(encoding="utf-8") as file:
    sample_line = file.readline().strip()
sample_image, sample_text = sample_line.split("\t")
print(f"Label: {sample_text}")
display(Image(filename=str(DATASET_DIR / sample_image)))

Label: 29A87180


## Fine-tune with PaddleOCR's tools/train.py (on GPU)

Same invocation validated locally in `scripts/train-ocr.py` (config discovery,
correct `-o` usage, absolute paths, `Global.use_gpu`, batch size override) --
just with `use_gpu=true` here since a GPU is available.

In [11]:
CONFIG_NAME = "PP-OCRv5_mobile_rec"
# New directory (not the old ocr-rec-training) so this run fine-tunes fresh from
# PRETRAINED_MODEL instead of auto-resuming from a checkpoint that was trained
# (and overfit) on the smaller, less diverse pre-merge dataset.
OUTPUT_DIR = "/content/drive/MyDrive/CITD/XuLyAnh/Doan/ocr-rec-training-merged"
EPOCHS = 40
BATCH_SIZE = 256
# A full 40-epoch run on the pre-merge dataset peaked at epoch 9 (val acc 99.74%)
# and then overfit for the remaining 31 epochs -- val acc drifted down to ~84%
# while train-batch acc stayed ~99-100%. PATIENCE stops training once this
# many epochs pass with no new best instead of burning GPU time past that
# point; EPOCHS stays as an upper-bound safety net.
PATIENCE = 8
PRETRAINED_MODEL = (
    "https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/"
    "PP-OCRv5_mobile_rec_pretrained.pdparams"
)

In [12]:
import re
import subprocess
import sys
from pathlib import Path

config_matches = sorted(REPO_DIR.glob(f"**/*{CONFIG_NAME}*.yml"))
assert config_matches, f"No config matching {CONFIG_NAME!r} found under {REPO_DIR}"
config_path = config_matches[0]
print(f"Using config: {config_path}")

dataset_dir = Path(DATASET_DIR).resolve()
output_dir = Path(OUTPUT_DIR).resolve()

iters_per_epoch = max(1, train_count // BATCH_SIZE)

overrides = [
    "Global.use_gpu=true",
    f"Global.save_model_dir={output_dir}",
    f"Train.dataset.data_dir={dataset_dir}",
    f"Train.dataset.label_file_list=[{dataset_dir}/train.txt]",
    f"Eval.dataset.data_dir={dataset_dir}",
    f"Eval.dataset.label_file_list=[{dataset_dir}/val.txt]",
    f"Global.epoch_num={EPOCHS}",
    f"Global.eval_batch_step=[0,{iters_per_epoch}]",
    f"Train.loader.batch_size_per_card={BATCH_SIZE}",
    f"Eval.loader.batch_size_per_card={BATCH_SIZE}",
    f"Train.sampler.first_bs={BATCH_SIZE}",
]

latest_checkpoint = output_dir / "latest"
if (output_dir / "latest.pdparams").is_file():
    print(f"Found existing checkpoint, resuming from: {latest_checkpoint}")
    overrides.append(f"Global.checkpoints={latest_checkpoint}")
else:
    print(f"No existing checkpoint; fine-tuning from pretrained: {PRETRAINED_MODEL}")
    overrides.append(f"Global.pretrained_model={PRETRAINED_MODEL}")

command = [sys.executable, "tools/train.py", "-c", str(config_path), "-o", *overrides]

# A prior 40-epoch run plateaued at epoch 9 (val acc 99.74%) then overfit --
# train-batch acc stayed ~99-100% but val acc drifted down into the 82-93%
# range for the remaining 31 epochs. PaddleOCR already checkpoints the best
# epoch separately (best_accuracy, see the export cell below), but training
# past that point just burns GPU time, so stop once PATIENCE epochs pass
# with no new best.
epoch_re = re.compile(r"epoch: \[(\d+)/\d+\]")
best_epoch_re = re.compile(r"best_epoch: (\d+)")
current_epoch = 0
best_epoch = 0
stopped_early = False

process = subprocess.Popen(
    command,
    cwd=REPO_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end="")
    epoch_match = epoch_re.search(line)
    if epoch_match:
        current_epoch = int(epoch_match.group(1))
    best_match = best_epoch_re.search(line)
    if best_match:
        best_epoch = int(best_match.group(1))
        if current_epoch - best_epoch >= PATIENCE:
            print(
                f"\nNo val improvement for {PATIENCE} epochs "
                f"(best_epoch={best_epoch}, current_epoch={current_epoch}); stopping early."
            )
            process.terminate()
            stopped_early = True
            break

returncode = process.wait()
if returncode != 0 and not stopped_early:
    raise subprocess.CalledProcessError(returncode, command)
print(f"Best epoch: {best_epoch}" + (" (stopped early)" if stopped_early else ""))

Using config: /usr/local/lib/python3.13/dist-packages/paddlex/repo_manager/repos/PaddleOCR/configs/rec/PP-OCRv5/PP-OCRv5_mobile_rec.yml
No existing checkpoint; fine-tuning from pretrained: https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_mobile_rec_pretrained.pdparams
/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Skipping import of the encryption module.
[2026/09/16 16:46:18] ppocr INFO: Architecture : 
[2026/09/16 16:46:18] ppocr INFO:     Backbone : 
[2026/09/16 16:46:18] ppocr INFO:         name : PPLCNetV3
[2026/09/16 16:46:18] ppocr INFO:         scale : 0.95
[2026/09/16 16:46:18] ppocr INFO:     Head : 
[2026/09/16 16:46:18] ppocr INFO:         head_list : 
[2026/09/16 16

## Export to PaddleOCR inference format

`best_accuracy.pdparams` (saved above) is a **training checkpoint** (dynamic
graph) -- `PaddleOCRBackend` in `src/lpr/ocr.py` (`PaddleOCR(text_recognition_model_dir=...)`)
needs the **inference format** instead (`inference.json`/`.pdiparams`/`.yml`,
produced by `tools/export_model.py`). This is what actually gets used to plug
the fine-tuned model into `lpr infer-image`/`lpr infer-video`.

In [ ]:
import subprocess
import sys
from pathlib import Path

output_dir = Path(OUTPUT_DIR).resolve()
best_checkpoint = output_dir / "best_accuracy.pdparams"
assert best_checkpoint.is_file(), f"No best_accuracy.pdparams found under {output_dir}"

INFERENCE_DIR = output_dir / "inference"

command = [
    sys.executable,
    "tools/export_model.py",
    "-c",
    str(config_path),
    "-o",
    f"Global.pretrained_model={best_checkpoint}",
    f"Global.save_inference_dir={INFERENCE_DIR}",
]
result = subprocess.run(command, cwd=REPO_DIR, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
result.check_returncode()

expected_files = {"inference.json", "inference.pdiparams", "inference.yml"}
found_files = {path.name for path in INFERENCE_DIR.iterdir()} if INFERENCE_DIR.is_dir() else set()
missing_files = expected_files - found_files
assert not missing_files, f"Export is missing {missing_files} (found: {found_files})"
print(f"Inference model exported to: {INFERENCE_DIR}")

## Save the fine-tuned model

Zips both the `best_accuracy.*` training checkpoint (for resuming/continuing
fine-tuning later) and the `inference/` export from the cell above (for
actual use), and copies the zip into a `model/` folder next to the dataset
on Drive (`.../Doan/model/ocr-rec-training-merged.zip`). `latest.*` (the
post-overfit checkpoint) is intentionally left out. After downloading and
unzipping locally, use the `inference/` subfolder with
`lpr infer-image --ocr paddleocr --rec-model-dir <unzipped-dir>/inference`.

In [ ]:
import shutil
from pathlib import Path

# PaddleOCR saves checkpoints as <prefix>.pdparams/.pdopt/.states directly under
# OUTPUT_DIR (prefix "latest", "best_accuracy", ...), not as subfolders. Stage
# the best_accuracy checkpoint (for resuming training later) plus the exported
# inference/ directory (for actual use) -- latest.* is the post-overfit
# checkpoint and has no reason to leave Colab.
output_dir = Path(OUTPUT_DIR).resolve()
best_files = sorted(output_dir.glob("best_accuracy.*"))
assert best_files, f"No best_accuracy.* checkpoint found under {output_dir}"
inference_dir = output_dir / "inference"
assert inference_dir.is_dir(), (
    f"No exported inference model at {inference_dir} -- run the "
    "export-to-inference-format cell above first"
)

staging_dir = Path("/content/ocr-rec-best")
if staging_dir.exists():
    shutil.rmtree(staging_dir)
(staging_dir / "checkpoint").mkdir(parents=True)
(staging_dir / "inference").mkdir(parents=True)
for file in best_files:
    shutil.copy(file, staging_dir / "checkpoint" / file.name)
config_yml = output_dir / "config.yml"
if config_yml.is_file():
    shutil.copy(config_yml, staging_dir / "checkpoint" / config_yml.name)
for file in inference_dir.iterdir():
    shutil.copy(file, staging_dir / "inference" / file.name)

archive_path = shutil.make_archive("/content/ocr-rec-training-merged", "zip", staging_dir)
print(archive_path)

MODEL_DRIVE_DIR = Path("/content/drive/MyDrive/CITD/XuLyAnh/Doan/model")
MODEL_DRIVE_DIR.mkdir(parents=True, exist_ok=True)
destination = MODEL_DRIVE_DIR / "ocr-rec-training-merged.zip"
shutil.copy(archive_path, destination)
print(f"Saved to: {destination}")

## Reproducibility checklist

Record in the final report: the dataset zip's origin and the commit SHA from
`.git-dataset-manifest.json` (see `docs/ocr-dataset-selection.md`), `CONFIG`,
`EPOCHS`, `PATIENCE`, `DEVICE`, the resulting `train.txt`/`val.txt` sample
counts printed above, the actual `best_epoch` printed by the training cell,
and the saved model path (the exported zip has `checkpoint/` -- the
`best_accuracy.*` training weights, for resuming fine-tuning later -- and
`inference/` -- the exported static model actually used by
`lpr infer-image --rec-model-dir`; `latest.*` is excluded entirely). The
"best metric, acc: ..." line PaddleOCR prints during training is its own
internal sequence-exact-match on the `val.txt` split -- not the project's
CER/exact accuracy/character accuracy. Do not claim OCR accuracy numbers
until a real held-out evaluation (`lpr evaluate-ocr`) has been run against
this fine-tuned model.

In [ ]:
# Only clear the raw checkpoint directory from Drive after the zip export
# above has actually completed -- otherwise this silently discards
# best_accuracy along with everything else, and ignore_errors=True was
# hiding any earlier failure to export.
import shutil
from pathlib import Path

output_dir = Path(OUTPUT_DIR).resolve()
assert destination.is_file(), (
    f"{destination} not found -- rerun the export cell above before deleting checkpoints"
)
shutil.rmtree(output_dir)
print(f"Removed {output_dir} (zip already saved at {destination})")